In [ ]:
%pip install pynput

In [2]:
import json, random, sys, threading, time, tkinter as tk
from collections import deque
from pathlib import Path

import numpy as np
from pynput import keyboard, mouse

In [3]:
WINDOW_S = 20                 # feature window length (seconds)
BASELINE_FILE = Path("baseline.json")
Z_THRESHOLD = 1.5             # a feature is "flagged" if it deviates this many SDs in the anxious direction
WINDOW_SCORE_THRESHOLD = 0.4  # fraction of features flagged for a window to count as "elevated"
MIN_FLAGGED = 3               # ...and at least this many features must be flagged (cuts one-off noise)
SUSTAINED = (3, 5)            # need 3 elevated windows out of the last 5 to raise the alert
N_TARGETS = 15

# feature -> direction expected under anxiety (+1 = increases, -1 = decreases)
DIRECTION = {
    "dwell_mean": +1, "flight_std": +1, "kpm": -1, "backspace_ratio": +1, "pause_ratio": +1,
    "speed_cv": +1, "directness": -1, "idle_ratio": +1, "scroll_reversals": +1, "coverage": +1,
}

GROUP = {n: "keyboard" for n in ("dwell_mean", "flight_std", "kpm", "backspace_ratio", "pause_ratio")}
GROUP.update({n: "mouse" for n in ("speed_cv", "directness", "idle_ratio", "scroll_reversals", "coverage")})
LABEL = {
    "dwell_mean": "key-hold time", "flight_std": "gap-between-keys variability", "kpm": "typing speed",
    "backspace_ratio": "backspace rate", "pause_ratio": "typing pauses",
    "speed_cv": "mouse speed variability", "directness": "mouse path directness",
    "idle_ratio": "mouse idle time", "scroll_reversals": "scroll reversals", "coverage": "screen coverage",
}

QUESTIONS = [
    "Describe what you had for breakfast today.",
    "Describe the route from your home to your university or workplace.",
    "Write a few sentences about your favourite hobby.",
    "Explain how to make a cup of tea or coffee.",
    "Describe the room you are sitting in right now.",
    "Write about a movie or show you enjoyed recently.",
]

In [4]:
class Collector:
    def __init__(self):
        self.lock = threading.Lock()
        self.down = {}
        self.keys, self.moves, self.scrolls = [], [], []   # all tuples start with a timestamp
        self.kl = keyboard.Listener(on_press=self._press, on_release=self._release)
        self.ml = mouse.Listener(on_move=self._move, on_scroll=self._scroll)

    def _press(self, key):
        self.down.setdefault(key, time.perf_counter())      # ignore auto-repeat

    def _release(self, key):
        t, d = time.perf_counter(), self.down.pop(key, None)
        if d is not None:
            with self.lock:
                self.keys.append((d, t, key == keyboard.Key.backspace))  # no key identity stored

    def _move(self, x, y):
        with self.lock:
            self.moves.append((time.perf_counter(), x, y))

    def _scroll(self, x, y, dx, dy):
        with self.lock:
            self.scrolls.append((time.perf_counter(), dy))

    def start(self):
        self.kl.start(); self.ml.start()

    def stop(self):
        self.kl.stop(); self.ml.stop()

    def drain(self):
        with self.lock:
            out = (self.keys, self.moves, self.scrolls)
            self.keys, self.moves, self.scrolls = [], [], []
        return out

In [5]:
def features(keys, moves, scrolls, dur, screen):
    f = {}
    if len(keys) >= 10:
        keys = sorted(keys)
        dwell = np.array([u - d for d, u, _ in keys])
        flight = np.array([keys[i + 1][0] - keys[i][1] for i in range(len(keys) - 1)])
        flight = flight[(flight > 0) & (flight < 2.0)]       # drop rollover overlaps and long pauses
        gaps = np.diff([k[0] for k in keys])
        f["dwell_mean"] = dwell.mean()
        f["kpm"] = len(keys) * 60.0 / dur
        f["backspace_ratio"] = sum(k[2] for k in keys) / len(keys)
        f["pause_ratio"] = float((gaps > 0.5).mean())         # stop-and-go typing
        if len(flight) >= 5:
            f["flight_std"] = flight.std()

    if len(moves) >= 20:
        t, x, y = np.array(moves).T
        dt, dist = np.diff(t), np.hypot(np.diff(x), np.diff(y))
        ok = dt > 0
        speed = dist[ok] / dt[ok]
        if len(speed) >= 10 and speed.mean() > 0:
            f["speed_cv"] = speed.std() / speed.mean()
        # directness per movement segment (split on pauses > 0.3 s)
        cuts = np.where(dt > 0.3)[0] + 1
        ratios = []
        for sx, sy in zip(np.split(x, cuts), np.split(y, cuts)):
            if len(sx) < 5:
                continue
            path = np.hypot(np.diff(sx), np.diff(sy)).sum()
            if path > 100:
                ratios.append(np.hypot(sx[-1] - sx[0], sy[-1] - sy[0]) / path)
        if ratios:
            f["directness"] = float(np.mean(ratios))
        f["idle_ratio"] = float(dt[dt > 0.5].sum() / dur)
        w, h = screen
        cells = {(int(a / w * 10), int(b / h * 10)) for a, b in zip(x, y)}
        f["coverage"] = len(cells) / 100.0

    if len(scrolls) >= 3:
        signs = np.sign([s[1] for s in scrolls if s[1] != 0])
        f["scroll_reversals"] = float((np.diff(signs) != 0).sum()) * 60.0 / dur
    return f


def chunk(data, t0, t1):
    return tuple([e for e in stream if t0 <= e[0] < t1] for stream in data)

In [6]:
def build_baseline(windows):
    base = {}
    for name in DIRECTION:
        vals = [w[name] for w in windows if name in w]
        if len(vals) >= 2:
            base[name] = [float(np.mean(vals)), float(np.std(vals))]
    return base


def score(f, base):
    """Per modality: [n_flagged, n_scored, [(feature, z), ...]]"""
    res = {"keyboard": [0, 0, []], "mouse": [0, 0, []]}
    for name, v in f.items():
        if name not in base:
            continue
        mu, sd = base[name]
        sd = max(sd, 0.15 * abs(mu), 1e-6)                    # floor so tiny baseline SD can't explode z
        z = DIRECTION[name] * (v - mu) / sd
        g = res[GROUP[name]]
        g[1] += 1
        if z > Z_THRESHOLD:
            g[0] += 1
            g[2].append((name, z))
    return res

In [7]:
def screen_size():
    r = tk.Tk(); wh = (r.winfo_screenwidth(), r.winfo_screenheight()); r.destroy()
    return wh


def run_calibration_ui():
    root = tk.Tk(); root.title("Calibration"); root.geometry("900x600")
    label = tk.Label(root, font=("Arial", 14), wraplength=850); label.pack(pady=10)
    box = tk.Text(root, height=6, font=("Arial", 13)); box.pack(fill="x", padx=20)
    canvas = tk.Canvas(root, bg="white")
    qs, hits = iter(random.sample(QUESTIONS, len(QUESTIONS))), [0]

    def spawn():
        if hits[0] >= N_TARGETS:
            root.destroy(); return
        canvas.delete("all")
        w, h = canvas.winfo_width(), canvas.winfo_height()
        x, y = random.randint(40, w - 40), random.randint(40, h - 40)
        canvas.create_oval(x - 20, y - 20, x + 20, y + 20, fill="steelblue", tags="t")
        canvas.tag_bind("t", "<Button-1>", hit)

    def hit(_):
        hits[0] += 1; spawn()

    def targets():
        box.pack_forget(); btn.pack_forget()
        canvas.pack(fill="both", expand=True)
        label.config(text=f"Click each circle ({N_TARGETS} total)")
        root.update(); spawn()

    def next_q():
        box.delete("1.0", "end")
        try:
            label.config(text=next(qs))
        except StopIteration:
            targets()

    btn = tk.Button(root, text="Next", command=next_q); btn.pack(pady=8)
    next_q(); root.mainloop()


def calibrate(passive_minutes=None):
    """Guided calibration (default), or passive: record `passive_minutes` of your normal computer use."""
    screen = screen_size()
    c = Collector(); c.start()
    t0 = time.perf_counter()
    if passive_minutes:
        print(f"Recording {passive_minutes} min of normal use... work as usual. Interrupt the kernel to cancel.")
        time.sleep(passive_minutes * 60)
    else:
        run_calibration_ui()
    t1 = time.perf_counter()
    c.stop()
    data = c.drain()
    starts = np.arange(t0, t1 - WINDOW_S + 1e-9, WINDOW_S)
    windows = [features(*chunk(data, s, s + WINDOW_S), WINDOW_S, screen) for s in starts]
    base = build_baseline(windows)
    if len(base) < 4:
        print("Not enough data for a baseline. Type more and move the mouse more, then retry.")
        return
    BASELINE_FILE.write_text(json.dumps(base, indent=2))
    print(f"Baseline saved ({len(base)} features from {len(windows)} windows).")

In [8]:
calibrate()

Baseline saved (9 features from 7 windows).


In [9]:
def monitor(max_windows=None):
    base = json.loads(BASELINE_FILE.read_text())
    screen = screen_size()
    c = Collector(); c.start()
    recent = deque(maxlen=SUSTAINED[1])
    counts, n_win, n_elev = {}, 0, 0
    print(f"Monitoring in {WINDOW_S}s windows. Interrupt the kernel (stop button) to end.\n"
          "Arrows show the direction the feature moved vs YOUR baseline; z = how many SDs away.\n")
    try:
        while max_windows is None or n_win < max_windows:
            time.sleep(WINDOW_S)
            f = features(*c.drain(), WINDOW_S, screen)
            res = score(f, base)
            total = sum(r[1] for r in res.values())
            stamp = time.strftime("%H:%M:%S")
            if total < 3:
                print(f"{stamp}  (skipped: too little keyboard/mouse activity)"); continue
            n_win += 1
            flagged = sum(r[0] for r in res.values())
            elevated = flagged >= MIN_FLAGGED and flagged / total >= WINDOW_SCORE_THRESHOLD
            recent.append(elevated); n_elev += elevated
            dots = "".join("●" if r else "○" for r in recent)
            print(f"{stamp}  window #{n_win}: {'ELEVATED' if elevated else 'normal  '}  "
                  f"({flagged}/{total} features shifted)   recent: {dots}")
            for grp, (nf, nt, items) in res.items():
                if nt == 0:
                    print(f"    {grp:<8} no data"); continue
                detail = "  ".join(f"{'↑' if DIRECTION[n] > 0 else '↓'} {LABEL[n]} (z={z:.1f})" for n, z in items)
                print(f"    {grp:<8} {nf}/{nt}  {detail}")
                for n, _ in items:
                    counts[n] = counts.get(n, 0) + 1
            if sum(recent) >= SUSTAINED[0]:
                print(f"    >>> SUSTAINED SHIFT: {sum(recent)} of the last {len(recent)} windows elevated")
    except KeyboardInterrupt:
        pass
    finally:
        c.stop()
        print(f"\nSummary: {n_elev}/{n_win} windows elevated.")
        for n, k in sorted(counts.items(), key=lambda x: -x[1]):
            print(f"  {GROUP[n]:<8} {LABEL[n]}: flagged in {k} window(s)")

In [10]:
monitor()

Monitoring in 20s windows. Interrupt the kernel (stop button) to end.
Arrows show the direction the feature moved vs YOUR baseline; z = how many SDs away.

00:08:45  window #1: normal    (2/4 features shifted)   recent: ○
    keyboard no data
    mouse    2/4  ↓ mouse path directness (z=3.0)  ↑ screen coverage (z=15.1)
00:09:05  window #2: normal    (1/9 features shifted)   recent: ○○
    keyboard 1/5  ↑ key-hold time (z=7.0)
    mouse    0/4  
00:09:25  window #3: normal    (1/5 features shifted)   recent: ○○○
    keyboard 1/5  ↑ key-hold time (z=9.9)
    mouse    no data
00:09:45  window #4: normal    (2/5 features shifted)   recent: ○○○○
    keyboard 2/5  ↑ key-hold time (z=15.8)  ↑ backspace rate (z=2.0)
    mouse    no data
00:10:05  window #5: normal    (2/4 features shifted)   recent: ○○○○○
    keyboard no data
    mouse    2/4  ↑ mouse idle time (z=4.1)  ↑ screen coverage (z=2.6)
00:10:25  window #6: normal    (1/4 features shifted)   recent: ○○○○○
    keyboard no data
    mous